# مشروع Database Replication على Google Colab
PostgreSQL Streaming Replication: سيرفر Primary (المنفذ 5432) + سيرفر Replica (المنفذ 5433) داخل نفس جلسة Colab.

شغّلي الخلايا بالترتيب (Shift+Enter).

In [13]:
%%bash
su postgres -c %%bash
su postgres -c "psql -p 5432 -d postgres -c 'CREATE TABLE IF NOT EXISTS ledger (id BIGINT PRIMARY KEY, ts TIMESTAMP);'"

CREATE TABLE


bash: line 1: fg: no job control


## 1) تثبيت PostgreSQL داخل Colab

In [1]:
%%bash
apt-get -qq update
apt-get -qq install -y postgresql postgresql-contrib > /dev/null
echo "تم التثبيت"

تم التثبيت


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os, subprocess
VER = sorted(os.listdir('/usr/lib/postgresql'))[-1]
os.environ['VER'] = VER
print('PostgreSQL version:', VER)

PostgreSQL version: 16


## 2) تشغيل الـ Primary وإنشاء مستخدم النسخ

In [38]:
%%writefile ledger_client.py
import time
import psycopg

# قمنا بتبسيط الـ DSN ليستخدم ميزم النظام المحلي الموثوق تلقائياً دون قيود الشبكة
DSN = "dbname=postgres user=postgres"

def main(duration=60):
    end = time.time() + duration
    i = 0

    with open('acked.log', 'w') as log:  # استخدمنا 'w' لتنظيف السجل القديم والبدء من جديد
        try:
            with psycopg.connect(DSN, autocommit=False) as conn:
                with conn.cursor() as cur:
                    while time.time() < end:
                        i += 1
                        try:
                            cur.execute("INSERT INTO ledger (id, ts) VALUES (%s, now())", (i,))
                            conn.commit()
                            log.write(f"{i}\n")
                            log.flush()
                        except Exception as e:
                            print(f"فشلت المعاملة {i}: {e}")
                            conn.rollback()
                            break
        except Exception as e:
            print(f"فشل الاتصال بقاعدة البيانات: {e}")

if __name__ == '__main__':
    main()

Overwriting ledger_client.py


In [35]:
%%bash
su - postgres -c "python3 /content/ledger_client.py"

فشل الاتصال بقاعدة البيانات: connection failed: connection to server at "127.0.0.1", port 5432 failed: fe_sendauth: no password supplied
Multiple connection attempts failed. All failures were:
- host: 'localhost', port: '5432', hostaddr: '::1': connection failed: connection to server at "::1", port 5432 failed: fe_sendauth: no password supplied
- host: 'localhost', port: '5432', hostaddr: '127.0.0.1': connection failed: connection to server at "127.0.0.1", port 5432 failed: fe_sendauth: no password supplied


In [36]:
%%bash
cp /content/ledger_client.py /tmp/ledger_client.py
chmod 777 /tmp/ledger_client.py

In [39]:
%%bash
cp /content/ledger_client.py /tmp/ledger_client.py
chmod 777 /tmp/ledger_client.py
su - postgres -c "python3 /tmp/ledger_client.py"

In [32]:
%%bash
python3 ledger_client.py

فشل الاتصال بقاعدة البيانات: connection failed: connection to server at "127.0.0.1", port 5432 failed: fe_sendauth: no password supplied
Multiple connection attempts failed. All failures were:
- host: 'localhost', port: '5432', hostaddr: '::1': connection failed: connection to server at "::1", port 5432 failed: fe_sendauth: no password supplied
- host: 'localhost', port: '5432', hostaddr: '127.0.0.1': connection failed: connection to server at "127.0.0.1", port 5432 failed: fe_sendauth: no password supplied


In [27]:
!pip install "psycopg[binary]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 17.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [psycopg]


In [30]:
%%bash
service postgresql start

 * Starting PostgreSQL 16 database server
   ...done.


In [3]:
%%bash
pg_ctlcluster $VER main start || true
sleep 3
su postgres -c "psql -c \"CREATE ROLE replicator WITH REPLICATION LOGIN PASSWORD 'rep123';\"" || true
su postgres -c "psql -c 'SHOW wal_level;'"

CREATE ROLE
 wal_level 
-----------
 replica
(1 row)



## 3) بناء الـ Replica من نسخة أساسية (pg_basebackup) وتشغيله

In [4]:
%%bash
pg_ctlcluster $VER replica stop 2>/dev/null || true
pg_dropcluster $VER replica 2>/dev/null || true
pg_createcluster $VER replica -p 5433
rm -rf /var/lib/postgresql/$VER/replica
su postgres -c "PGPASSWORD=rep123 pg_basebackup -h 127.0.0.1 -p 5432 -U replicator -D /var/lib/postgresql/$VER/replica -R -X stream -P"
pg_ctlcluster $VER replica start
sleep 3
pg_lsclusters

Creating new PostgreSQL cluster 16/replica ...
/usr/lib/postgresql/16/bin/initdb -D /var/lib/postgresql/16/replica --auth-local peer --auth-host scram-sha-256 --no-instructions
The files belonging to this database system will be owned by user "postgres".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default database encoding has accordingly been set to "UTF8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

fixing permissions on existing directory /var/lib/postgresql/16/replica ... ok
creating subdirectories ... ok
selecting dynamic shared memory implementation ... posix
selecting default max_connections ... 100
selecting default shared_buffers ... 128MB
selecting default time zone ... Etc/UTC
creating configuration files ... ok
running bootstrap script ... ok
performing post-bootstrap initialization ... ok
syncing data to disk ... ok
Ver Cluster Port Status Owner 

waiting for checkpoint
23094/23094 kB (100%), 0/1 tablespace
23094/23094 kB (100%), 1/1 tablespace


## 4) التحقق من حالة النسخ

In [5]:
%%bash
echo "--- على الـ Primary: الـ Replicas المتصلة ---"
su postgres -c "psql -p 5432 -c 'SELECT client_addr, state, sync_state FROM pg_stat_replication;'"
echo "--- على الـ Replica: هل هو بوضع استعادة؟ (t = نعم) ---"
su postgres -c "psql -p 5433 -c 'SELECT pg_is_in_recovery();'"

--- على الـ Primary: الـ Replicas المتصلة ---
 client_addr |   state   | sync_state 
-------------+-----------+------------
 127.0.0.1   | streaming | async
(1 row)

--- على الـ Replica: هل هو بوضع استعادة؟ (t = نعم) ---
 pg_is_in_recovery 
-------------------
 t
(1 row)



## 5) اختبار: إدخال بيانات على Primary وقراءتها من Replica

In [6]:
%%bash
su postgres -c "psql -p 5432 -c 'CREATE TABLE IF NOT EXISTS students(id SERIAL PRIMARY KEY, name TEXT);'"
su postgres -c "psql -p 5432 -c \"INSERT INTO students(name) VALUES ('Ali'),('Sara'),('Omar');\""
sleep 2
echo "--- القراءة من الـ Replica ---"
su postgres -c "psql -p 5433 -c 'SELECT * FROM students;'"

CREATE TABLE
INSERT 0 3
--- القراءة من الـ Replica ---
 id | name 
----+------
  1 | Ali
  2 | Sara
  3 | Omar
(3 rows)



## 6) اختبار: الكتابة على الـ Replica مرفوضة (Read-Only)

In [7]:
%%bash
su postgres -c "psql -p 5433 -c \"INSERT INTO students(name) VALUES ('Test');\"" || echo "متوقع: الـ Replica للقراءة فقط"

متوقع: الـ Replica للقراءة فقط


ERROR:  cannot execute INSERT in a read-only transaction


## 7) محاكاة الفشل (Failover): إيقاف الـ Primary وترقية الـ Replica

In [10]:
%%bash
pg_ctlcluster 16 main start
sleep 3
su postgres -c "psql -p 5433 -c 'SELECT pg_is_in_recovery();'"
su postgres -c "psql -p 5433 -c \"INSERT INTO students(name) VALUES ('NewPrimaryTest');\""
su postgres -c "psql -p 5433 -c 'SELECT * FROM students;'"

 pg_is_in_recovery 
-------------------
 f
(1 row)

INSERT 0 1
 id |      name      
----+----------------
  1 | Ali
  2 | Sara
  3 | Omar
 34 | NewPrimaryTest
 35 | NewPrimaryTest
(5 rows)



In [40]:
%%bash
# 1. معرفة آخر رقم معاملة تم إدخاله في جدول قاعدة البيانات
su - postgres -c "psql -d postgres -c 'SELECT MAX(id), COUNT(*) FROM ledger;'"

# 2. معرفة آخر رقم معاملة تم تأكيده وحفظه داخل ملف acked.log
tail -n 5 acked.log

  max  | count 
-------+-------
 24230 | 24230
(1 row)



In [42]:
%%bash
# 1. فحص ملف السجل للتأكد من المعاملات المؤكدة
tail -n 5 acked.log

# 2. الاستعلام عن حالة السيرفرات الاحتياطية المتصلة بالأساسي
su - postgres -c "psql -d postgres -c 'SELECT application_name, state, sync_state, sync_priority FROM pg_stat_replication;'"

 application_name | state | sync_state | sync_priority 
------------------+-------+------------+---------------
(0 rows)



In [12]:
%%bash
# 1. مسح أي مجلد قديم لـ standby1 إن وجد لضمان بناء نظيف
rm -rf /var/lib/postgresql/16/standby1

# 2. أخذ نسخة أساسية من السيرفر الرئيسي (Port 5432) وتهيئتها كـ Standby متصل تلقائياً
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -D /var/lib/postgresql/16/standby1 -R -P -U postgres"

# 3. تعديل منفذ السيرفر الاحتياطي الأول ليعمل على المنفذ 5433 لكي لا يتعارض مع الرئيسي
echo "port = 5433" >> /var/lib/postgresql/16/standby1/postgresql.conf

# 4. تشغيل السيرفر الاحتياطي الأول بنجاح
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

pg_basebackup: error: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
bash: line 8: /var/lib/postgresql/16/standby1/postgresql.conf: No such file or directory
pg_ctl: directory "/var/lib/postgresql/16/standby1" does not exist


CalledProcessError: Command 'b'# 1. \xd9\x85\xd8\xb3\xd8\xad \xd8\xa3\xd9\x8a \xd9\x85\xd8\xac\xd9\x84\xd8\xaf \xd9\x82\xd8\xaf\xd9\x8a\xd9\x85 \xd9\x84\xd9\x80 standby1 \xd8\xa5\xd9\x86 \xd9\x88\xd8\xac\xd8\xaf \xd9\x84\xd8\xb6\xd9\x85\xd8\xa7\xd9\x86 \xd8\xa8\xd9\x86\xd8\xa7\xd8\xa1 \xd9\x86\xd8\xb8\xd9\x8a\xd9\x81\nrm -rf /var/lib/postgresql/16/standby1\n\n# 2. \xd8\xa3\xd8\xae\xd8\xb0 \xd9\x86\xd8\xb3\xd8\xae\xd8\xa9 \xd8\xa3\xd8\xb3\xd8\xa7\xd8\xb3\xd9\x8a\xd8\xa9 \xd9\x85\xd9\x86 \xd8\xa7\xd9\x84\xd8\xb3\xd9\x8a\xd8\xb1\xd9\x81\xd8\xb1 \xd8\xa7\xd9\x84\xd8\xb1\xd8\xa6\xd9\x8a\xd8\xb3\xd9\x8a (Port 5432) \xd9\x88\xd8\xaa\xd9\x87\xd9\x8a\xd8\xa6\xd8\xaa\xd9\x87\xd8\xa7 \xd9\x83\xd9\x80 Standby \xd9\x85\xd8\xaa\xd8\xb5\xd9\x84 \xd8\xaa\xd9\x84\xd9\x82\xd8\xa7\xd8\xa6\xd9\x8a\xd8\xa7\xd9\x8b\nsu - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -D /var/lib/postgresql/16/standby1 -R -P -U postgres"\n\n# 3. \xd8\xaa\xd8\xb9\xd8\xaf\xd9\x8a\xd9\x84 \xd9\x85\xd

In [13]:
%%bash
# 1. إيقاف خادم الـ Replica لضمان إغلاق أي عملية معلقة
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true

# 2. مسح مجلد الـ Replica القديم بالكامل لتنظيف البيئة
rm -rf /var/lib/postgresql/16/standby1

# 3. سحب نسخة أساسية نظيفة ومتزامنة من السيرفر الرئيسي (بورت 5432) إلى مجلد الـ Replica
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 4. تشغيل خادم الـ Replica من جديد
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

pg_ctl: directory "/var/lib/postgresql/16/standby1" does not exist
pg_basebackup: error: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
pg_ctl: directory "/var/lib/postgresql/16/standby1" does not exist


CalledProcessError: Command 'b'# 1. \xd8\xa5\xd9\x8a\xd9\x82\xd8\xa7\xd9\x81 \xd8\xae\xd8\xa7\xd8\xaf\xd9\x85 \xd8\xa7\xd9\x84\xd9\x80 Replica \xd9\x84\xd8\xb6\xd9\x85\xd8\xa7\xd9\x86 \xd8\xa5\xd8\xba\xd9\x84\xd8\xa7\xd9\x82 \xd8\xa3\xd9\x8a \xd8\xb9\xd9\x85\xd9\x84\xd9\x8a\xd8\xa9 \xd9\x85\xd8\xb9\xd9\x84\xd9\x82\xd8\xa9\nsu - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true\n\n# 2. \xd9\x85\xd8\xb3\xd8\xad \xd9\x85\xd8\xac\xd9\x84\xd8\xaf \xd8\xa7\xd9\x84\xd9\x80 Replica \xd8\xa7\xd9\x84\xd9\x82\xd8\xaf\xd9\x8a\xd9\x85 \xd8\xa8\xd8\xa7\xd9\x84\xd9\x83\xd8\xa7\xd9\x85\xd9\x84 \xd9\x84\xd8\xaa\xd9\x86\xd8\xb8\xd9\x8a\xd9\x81 \xd8\xa7\xd9\x84\xd8\xa8\xd9\x8a\xd8\xa6\xd8\xa9\nrm -rf /var/lib/postgresql/16/standby1\n\n# 3. \xd8\xb3\xd8\xad\xd8\xa8 \xd9\x86\xd8\xb3\xd8\xae\xd8\xa9 \xd8\xa3\xd8\xb3\xd8\xa7\xd8\xb3\xd9\x8a\xd8\xa9 \xd9\x86\xd8\xb8\xd9\x8a\xd9\x81\xd8\xa9 \xd9\x88\xd9\x85\xd8\xaa\xd8\xb2\xd8\xa7\xd9\x85\xd9\x86\xd8\xa9 \xd9\x85\x

In [14]:
%%bash
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"

waiting for server to start....postgres: could not access the server configuration file "/var/lib/postgresql/16/main/postgresql.conf": No such file or directory
 stopped waiting


pg_ctl: could not start server
Examine the log output.


CalledProcessError: Command 'b'su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"\n'' returned non-zero exit status 1.

In [15]:
%%bash
# 1. إصلاح الصلاحيات وإعادة ملكية المجلدات للمستخدم postgres
chown -R postgres:postgres /var/lib/postgresql/16/main
chmod 700 /var/lib/postgresql/16/main

# 2. تشغيل السيرفر الرئيسي مجدداً
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"

waiting for server to start....postgres: could not access the server configuration file "/var/lib/postgresql/16/main/postgresql.conf": No such file or directory
 stopped waiting


pg_ctl: could not start server
Examine the log output.


CalledProcessError: Command 'b'# 1. \xd8\xa5\xd8\xb5\xd9\x84\xd8\xa7\xd8\xad \xd8\xa7\xd9\x84\xd8\xb5\xd9\x84\xd8\xa7\xd8\xad\xd9\x8a\xd8\xa7\xd8\xaa \xd9\x88\xd8\xa5\xd8\xb9\xd8\xa7\xd8\xaf\xd8\xa9 \xd9\x85\xd9\x84\xd9\x83\xd9\x8a\xd8\xa9 \xd8\xa7\xd9\x84\xd9\x85\xd8\xac\xd9\x84\xd8\xaf\xd8\xa7\xd8\xaa \xd9\x84\xd9\x84\xd9\x85\xd8\xb3\xd8\xaa\xd8\xae\xd8\xaf\xd9\x85 postgres\nchown -R postgres:postgres /var/lib/postgresql/16/main\nchmod 700 /var/lib/postgresql/16/main\n\n# 2. \xd8\xaa\xd8\xb4\xd8\xba\xd9\x8a\xd9\x84 \xd8\xa7\xd9\x84\xd8\xb3\xd9\x8a\xd8\xb1\xd9\x81\xd8\xb1 \xd8\xa7\xd9\x84\xd8\xb1\xd8\xa6\xd9\x8a\xd8\xb3\xd9\x8a \xd9\x85\xd8\xac\xd8\xaf\xd8\xaf\xd8\xa7\xd9\x8b\nsu - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"\n'' returned non-zero exit status 1.

In [1]:
%%bash
# تشغيل جميع العمليات بصلاحيات مستخدم postgres لتفادي قيود النظام
su - postgres -c "
# 1. إيقاف السيرفر الأساسي إجبارياً لإغلاق أي اتصالات معلقة
/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main stop -m immediate || true

# 2. تطبيق صلاحيات الثقة الكاملة لتخطي كلمات المرور
echo 'local   all             all                                     trust' > /var/lib/postgresql/16/main/pg_hba.conf
echo 'host    all             all             127.0.0.1/32            trust' >> /var/lib/postgresql/16/main/pg_hba.conf
echo 'host    replication     all             127.0.0.1/32            trust' >> /var/lib/postgresql/16/main/pg_hba.conf

# 3. إعادة تشغيل السيرفر الأساسي ونظامه
/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start

# 4. بناء مجلد السيرفر الاحتياطي الأول وتشغيله تلقائياً بدون كلمة مرور
rm -rf /var/lib/postgresql/16/standby1
/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -D /var/lib/postgresql/16/standby1 -R -P -U postgres
echo 'port = 5433' >> /var/lib/postgresql/16/standby1/postgresql.conf
/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start
"

waiting for server to start....postgres: could not access the server configuration file "/var/lib/postgresql/16/main/postgresql.conf": No such file or directory
 stopped waiting


pg_ctl: PID file "/var/lib/postgresql/16/main/postmaster.pid" does not exist
Is server running?
-bash: line 6: /var/lib/postgresql/16/main/pg_hba.conf: Permission denied
-bash: line 7: /var/lib/postgresql/16/main/pg_hba.conf: Permission denied
-bash: line 8: /var/lib/postgresql/16/main/pg_hba.conf: Permission denied
pg_ctl: could not start server
Examine the log output.
pg_basebackup: error: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
-bash: line 16: /var/lib/postgresql/16/standby1/postgresql.conf: No such file or directory
pg_ctl: directory "/var/lib/postgresql/16/standby1" does not exist


CalledProcessError: Command 'b'# \xd8\xaa\xd8\xb4\xd8\xba\xd9\x8a\xd9\x84 \xd8\xac\xd9\x85\xd9\x8a\xd8\xb9 \xd8\xa7\xd9\x84\xd8\xb9\xd9\x85\xd9\x84\xd9\x8a\xd8\xa7\xd8\xaa \xd8\xa8\xd8\xb5\xd9\x84\xd8\xa7\xd8\xad\xd9\x8a\xd8\xa7\xd8\xaa \xd9\x85\xd8\xb3\xd8\xaa\xd8\xae\xd8\xaf\xd9\x85 postgres \xd9\x84\xd8\xaa\xd9\x81\xd8\xa7\xd8\xaf\xd9\x8a \xd9\x82\xd9\x8a\xd9\x88\xd8\xaf \xd8\xa7\xd9\x84\xd9\x86\xd8\xb8\xd8\xa7\xd9\x85\nsu - postgres -c "\n# 1. \xd8\xa5\xd9\x8a\xd9\x82\xd8\xa7\xd9\x81 \xd8\xa7\xd9\x84\xd8\xb3\xd9\x8a\xd8\xb1\xd9\x81\xd8\xb1 \xd8\xa7\xd9\x84\xd8\xa3\xd8\xb3\xd8\xa7\xd8\xb3\xd9\x8a \xd8\xa5\xd8\xac\xd8\xa8\xd8\xa7\xd8\xb1\xd9\x8a\xd8\xa7\xd9\x8b \xd9\x84\xd8\xa5\xd8\xba\xd9\x84\xd8\xa7\xd9\x82 \xd8\xa3\xd9\x8a \xd8\xa7\xd8\xaa\xd8\xb5\xd8\xa7\xd9\x84\xd8\xa7\xd8\xaa \xd9\x85\xd8\xb9\xd9\x84\xd9\x82\xd8\xa9\n/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main stop -m immediate || true\n\n# 2. \xd8\xaa\xd8\xb7\xd8\xa8\xd9\x8a\xd9\x82 \xd8\xb5\xd9\x84\xd8\xa

In [11]:
%%bash
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"
su - postgres -c "rm -rf /var/lib/postgresql/16/standby1"
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -D /var/lib/postgresql/16/standby1 -R"
su - postgres -c "echo 'port = 5433' >> /var/lib/postgresql/16/standby1/postgresql.conf"
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

waiting for server to start....postgres: could not access the server configuration file "/var/lib/postgresql/16/main/postgresql.conf": No such file or directory
 stopped waiting


pg_ctl: could not start server
Examine the log output.
pg_basebackup: error: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
-bash: line 1: /var/lib/postgresql/16/standby1/postgresql.conf: No such file or directory
pg_ctl: directory "/var/lib/postgresql/16/standby1" does not exist


CalledProcessError: Command 'b'su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"\nsu - postgres -c "rm -rf /var/lib/postgresql/16/standby1"\nsu - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -D /var/lib/postgresql/16/standby1 -R"\nsu - postgres -c "echo \'port = 5433\' >> /var/lib/postgresql/16/standby1/postgresql.conf"\nsu - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"\n'' returned non-zero exit status 1.

In [2]:
%%bash
# 1. إرجاع ملكية مجلدات PostgreSQL بالكامل للمستخدم postgres
chown -R postgres:postgres /var/lib/postgresql/16/

# 2. ضبط الصلاحيات الأمنية الصحيحة للمجلدات لتشغيل السيرفر بدون قيود
chmod 700 /var/lib/postgresql/16/main
chmod 700 /var/lib/postgresql/16/standby1 || true

chmod: cannot access '/var/lib/postgresql/16/standby1': No such file or directory


In [4]:
%%bash
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main status"

pg_ctl: no server running


CalledProcessError: Command 'b'su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main status"\n'' returned non-zero exit status 3.

In [6]:
%%bash
# 1. مسح ملفات القفل المعلقة التي تمنع السيرفر من الإقلاع
rm -f /var/lib/postgresql/16/main/postmaster.pid
rm -f /tmp/.s.PGSQL.5432*

# 2. التأكد من الملكية الصحيحة للمجلد
chown -R postgres:postgres /var/lib/postgresql/16/main
chmod 700 /var/lib/postgresql/16/main

# 3. تشغيل السيرفر الرئيسي مجدداً بشكل صحيح
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"

waiting for server to start....postgres: could not access the server configuration file "/var/lib/postgresql/16/main/postgresql.conf": No such file or directory
 stopped waiting


pg_ctl: could not start server
Examine the log output.


CalledProcessError: Command 'b'# 1. \xd9\x85\xd8\xb3\xd8\xad \xd9\x85\xd9\x84\xd9\x81\xd8\xa7\xd8\xaa \xd8\xa7\xd9\x84\xd9\x82\xd9\x81\xd9\x84 \xd8\xa7\xd9\x84\xd9\x85\xd8\xb9\xd9\x84\xd9\x82\xd8\xa9 \xd8\xa7\xd9\x84\xd8\xaa\xd9\x8a \xd8\xaa\xd9\x85\xd9\x86\xd8\xb9 \xd8\xa7\xd9\x84\xd8\xb3\xd9\x8a\xd8\xb1\xd9\x81\xd8\xb1 \xd9\x85\xd9\x86 \xd8\xa7\xd9\x84\xd8\xa5\xd9\x82\xd9\x84\xd8\xa7\xd8\xb9\nrm -f /var/lib/postgresql/16/main/postmaster.pid\nrm -f /tmp/.s.PGSQL.5432*\n\n# 2. \xd8\xa7\xd9\x84\xd8\xaa\xd8\xa3\xd9\x83\xd8\xaf \xd9\x85\xd9\x86 \xd8\xa7\xd9\x84\xd9\x85\xd9\x84\xd9\x83\xd9\x8a\xd8\xa9 \xd8\xa7\xd9\x84\xd8\xb5\xd8\xad\xd9\x8a\xd8\xad\xd8\xa9 \xd9\x84\xd9\x84\xd9\x85\xd8\xac\xd9\x84\xd8\xaf\nchown -R postgres:postgres /var/lib/postgresql/16/main\nchmod 700 /var/lib/postgresql/16/main\n\n# 3. \xd8\xaa\xd8\xb4\xd8\xba\xd9\x8a\xd9\x84 \xd8\xa7\xd9\x84\xd8\xb3\xd9\x8a\xd8\xb1\xd9\x81\xd8\xb1 \xd8\xa7\xd9\x84\xd8\xb1\xd8\xa6\xd9\x8a\xd8\xb3\xd9\x8a \xd9\x85\xd8\xac\xd8\xaf\xd8\xa

In [7]:
%%bash
su - postgres -c "/usr/lib/postgresql/16/bin/postgres -D /var/lib/postgresql/16/main"

postgres: could not access the server configuration file "/var/lib/postgresql/16/main/postgresql.conf": No such file or directory


CalledProcessError: Command 'b'su - postgres -c "/usr/lib/postgresql/16/bin/postgres -D /var/lib/postgresql/16/main"\n'' returned non-zero exit status 2.

In [8]:
%%bash
# 1. إنشاء المجلد الرئيسي مجدداً في حال كان محذوفاً
mkdir -p /var/lib/postgresql/16/main
chown -R postgres:postgres /var/lib/postgresql/16/main
chmod 700 /var/lib/postgresql/16/main

# 2. إعادة تهيئة ملفات الإعدادات والبيانات الأساسية (initdb)
su - postgres -c "/usr/lib/postgresql/16/bin/initdb -D /var/lib/postgresql/16/main"

# 3. تشغيل السيرفر الرئيسي مجدداً بشكل صحيح
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"

The files belonging to this database system will be owned by user "postgres".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default database encoding has accordingly been set to "UTF8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

waiting for server to start....postgres: could not access the server configuration file "/var/lib/postgresql/16/main/postgresql.conf": No such file or directory
 stopped waiting


initdb: error: directory "/var/lib/postgresql/16/main" exists but is not empty
initdb: hint: If you want to create a new database system, either remove or empty the directory "/var/lib/postgresql/16/main" or run initdb with an argument other than "/var/lib/postgresql/16/main".
pg_ctl: could not start server
Examine the log output.


CalledProcessError: Command 'b'# 1. \xd8\xa5\xd9\x86\xd8\xb4\xd8\xa7\xd8\xa1 \xd8\xa7\xd9\x84\xd9\x85\xd8\xac\xd9\x84\xd8\xaf \xd8\xa7\xd9\x84\xd8\xb1\xd8\xa6\xd9\x8a\xd8\xb3\xd9\x8a \xd9\x85\xd8\xac\xd8\xaf\xd8\xaf\xd8\xa7\xd9\x8b \xd9\x81\xd9\x8a \xd8\xad\xd8\xa7\xd9\x84 \xd9\x83\xd8\xa7\xd9\x86 \xd9\x85\xd8\xad\xd8\xb0\xd9\x88\xd9\x81\xd8\xa7\xd9\x8b\nmkdir -p /var/lib/postgresql/16/main\nchown -R postgres:postgres /var/lib/postgresql/16/main\nchmod 700 /var/lib/postgresql/16/main\n\n# 2. \xd8\xa5\xd8\xb9\xd8\xa7\xd8\xaf\xd8\xa9 \xd8\xaa\xd9\x87\xd9\x8a\xd8\xa6\xd8\xa9 \xd9\x85\xd9\x84\xd9\x81\xd8\xa7\xd8\xaa \xd8\xa7\xd9\x84\xd8\xa5\xd8\xb9\xd8\xaf\xd8\xa7\xd8\xaf\xd8\xa7\xd8\xaa \xd9\x88\xd8\xa7\xd9\x84\xd8\xa8\xd9\x8a\xd8\xa7\xd9\x86\xd8\xa7\xd8\xaa \xd8\xa7\xd9\x84\xd8\xa3\xd8\xb3\xd8\xa7\xd8\xb3\xd9\x8a\xd8\xa9 (initdb)\nsu - postgres -c "/usr/lib/postgresql/16/bin/initdb -D /var/lib/postgresql/16/main"\n\n# 3. \xd8\xaa\xd8\xb4\xd8\xba\xd9\x8a\xd9\x84 \xd8\xa7\xd9\x84\xd8\xb3\x

In [9]:
%%bash
su - postgres -c "/usr/lib/postgresql/16/bin/postgres -D /var/lib/postgresql/16/main"

postgres: could not access the server configuration file "/var/lib/postgresql/16/main/postgresql.conf": No such file or directory


CalledProcessError: Command 'b'su - postgres -c "/usr/lib/postgresql/16/bin/postgres -D /var/lib/postgresql/16/main"\n'' returned non-zero exit status 2.

In [10]:
%%bash
# 1. تنظيف شامل للمجلدات لضمان عدم وجود ملفات معلقة
rm -rf /var/lib/postgresql/16/main
mkdir -p /var/lib/postgresql/16/main
chown -R postgres:postgres /var/lib/postgresql/16/main
chmod 700 /var/lib/postgresql/16/main

# 2. إعادة تهيئة قاعدة البيانات بالكامل كـ postgres
su - postgres -c "/usr/lib/postgresql/16/bin/initdb -D /var/lib/postgresql/16/main"

# 3. تشغيل السيرفر الرئيسي
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"

# 4. فحص الحالة النهائي للتأكد بنسبة 100%
echo "----------------------------------------"
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main status"

Process is interrupted.


In [11]:
%%bash
su - postgres -c "/usr/lib/postgresql/16/bin/postgres -D /var/lib/postgresql/16/main"

2026-10-27 03:11:48.656 UTC [55885] FATAL:  lock file "postmaster.pid" already exists
2026-10-27 03:11:48.656 UTC [55885] HINT:  Is another postmaster (PID 55170) running in data directory "/var/lib/postgresql/16/main"?


CalledProcessError: Command 'b'su - postgres -c "/usr/lib/postgresql/16/bin/postgres -D /var/lib/postgresql/16/main"\n'' returned non-zero exit status 1.

In [12]:
%%bash
# 1. إنشاء مستخدم التكرار بالصلاحيات اللازمة
su - postgres -c "psql -c \"CREATE ROLE replicator WITH REPLICATION LOGIN PASSWORD 'rep123';\"" 2>/dev/null || echo "المستخدم موجود مسبقاً أو تم إنشاؤه"

# 2. إنشاء جدول ledger داخل قاعدة البيانات الرئيسية
su - postgres -c "psql -d postgres -c \"CREATE TABLE IF NOT EXISTS ledger (id BIGINT PRIMARY KEY, ts TIMESTAMP);\""

CREATE ROLE
CREATE TABLE


In [13]:
%%bash
# 1. إيقاف خادم الـ Replica لضمان عدم وجود أي عملية قديمة معلقة
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true

# 2. تنظيف مجلد الـ Replica القديم بالكامل
rm -rf /var/lib/postgresql/16/standby1

# 3. سحب النسخة الاحتياطية المتزامنة من السيرفر الرئيسي (بورت 5432) إلى مجلد الـ Replica
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 4. تشغيل خادم الـ Replica من جديد (على بورت 5433 ليكون منفصلاً)
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 -o '-p 5433' start"

# 5. الانتظار ثانيتين ثم فحص حالة التكرار من السيرفر الرئيسي لنتأكد من الربط
sleep 2
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

waiting for server to start....2026-10-27 03:14:23.829 UTC [56541] LOG:  starting PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
2026-10-27 03:14:23.829 UTC [56541] LOG:  could not bind IPv6 address "::1": Address already in use
2026-10-27 03:14:23.829 UTC [56541] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:14:23.829 UTC [56541] LOG:  could not bind IPv4 address "127.0.0.1": Address already in use
2026-10-27 03:14:23.829 UTC [56541] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:14:23.829 UTC [56541] WARNING:  could not create listen socket for "localhost"
2026-10-27 03:14:23.829 UTC [56541] FATAL:  could not create any TCP/IP sockets
2026-10-27 03:14:23.832 UTC [56541] LOG:  database system is shut down
 stopped waiting
=== حالة التكرار النهائية ===
 application_name 

pg_ctl: directory "/var/lib/postgresql/16/standby1" does not exist
pg_ctl: could not start server
Examine the log output.


In [14]:
%%bash
# 1. إيقاف أي عملية قديمة معلقة للـ Replica تماماً
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true

# 2. تهيئة المجلد بالصلاحيات الصحيحة قبل السحب لضمان عدم الفشل
rm -rf /var/lib/postgresql/16/standby1
mkdir -p /var/lib/postgresql/16/standby1
chown -R postgres:postgres /var/lib/postgresql/16/standby1
chmod 700 /var/lib/postgresql/16/standby1
rm -rf /var/lib/postgresql/16/standby1

# 3. سحب النسخة المتزامنة النظيفة من السيرفر الرئيسي
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 4. تشغيل خادم الـ Replica على بورت منفصل تماماً (5433)
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 -o '-p 5433' start"

# 5. الانتظار لثوانٍ ثم فحص حالة التكرار النهائية من السيرفر الرئيسي
sleep 3
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

waiting for server to start....2026-10-27 03:16:21.782 UTC [57043] LOG:  starting PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
2026-10-27 03:16:21.783 UTC [57043] LOG:  could not bind IPv6 address "::1": Address already in use
2026-10-27 03:16:21.783 UTC [57043] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:16:21.783 UTC [57043] LOG:  could not bind IPv4 address "127.0.0.1": Address already in use
2026-10-27 03:16:21.783 UTC [57043] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:16:21.783 UTC [57043] WARNING:  could not create listen socket for "localhost"
2026-10-27 03:16:21.783 UTC [57043] FATAL:  could not create any TCP/IP sockets
2026-10-27 03:16:21.784 UTC [57043] LOG:  database system is shut down
 stopped waiting
=== حالة التكرار النهائية ===
 application_name 

pg_ctl: PID file "/var/lib/postgresql/16/standby1/postmaster.pid" does not exist
Is server running?
pg_ctl: could not start server
Examine the log output.


In [15]:
%%bash
# 1. إيقاف أي عملية قديمة معلقة للـ Replica
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true

# 2. تنظيف وسحب نسخة متزامنة نظيفة من السيرفر الرئيسي
rm -rf /var/lib/postgresql/16/standby1
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 3. تعديل ملف الإعدادات داخل مجلد الـ Replica لإجبارها على العمل على بورت 5433
echo "port = 5433" >> /var/lib/postgresql/16/standby1/postgresql.conf

# 4. تشغيل خادم الـ Replica الآن بأمان تام
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

# 5. الانتظار لثوانٍ ثم فحص حالة التكرار النهائية من السيرفر الرئيسي
sleep 3
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

waiting for server to start....2026-10-27 03:18:09.562 UTC [57500] LOG:  starting PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
2026-10-27 03:18:09.562 UTC [57500] LOG:  could not bind IPv6 address "::1": Address already in use
2026-10-27 03:18:09.562 UTC [57500] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:18:09.562 UTC [57500] LOG:  could not bind IPv4 address "127.0.0.1": Address already in use
2026-10-27 03:18:09.562 UTC [57500] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:18:09.562 UTC [57500] WARNING:  could not create listen socket for "localhost"
2026-10-27 03:18:09.562 UTC [57500] FATAL:  could not create any TCP/IP sockets
2026-10-27 03:18:09.564 UTC [57500] LOG:  database system is shut down
 stopped waiting
=== حالة التكرار النهائية ===
 application_name 

pg_ctl: PID file "/var/lib/postgresql/16/standby1/postmaster.pid" does not exist
Is server running?
pg_ctl: could not start server
Examine the log output.


In [16]:
%%bash
# 1. إيقاف أي عملية قديمة معلقة للـ Replica
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true

# 2. تنظيف وسحب نسخة متزامنة نظيفة من السيرفر الرئيسي
rm -rf /var/lib/postgresql/16/standby1
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 3. تشغيل خادم الـ Replica مع تمرير البورت 5433 مباشرة داخل أمر التشغيل لمنع التصادم
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 -o '-p 5433' start"

# 4. الانتظار لثوانٍ ثم فحص حالة التكرار النهائية من السيرفر الرئيسي
sleep 3
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

waiting for server to start....2026-10-27 03:19:44.619 UTC [57908] LOG:  starting PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
2026-10-27 03:19:44.619 UTC [57908] LOG:  could not bind IPv6 address "::1": Address already in use
2026-10-27 03:19:44.619 UTC [57908] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:19:44.620 UTC [57908] LOG:  could not bind IPv4 address "127.0.0.1": Address already in use
2026-10-27 03:19:44.620 UTC [57908] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:19:44.620 UTC [57908] WARNING:  could not create listen socket for "localhost"
2026-10-27 03:19:44.620 UTC [57908] FATAL:  could not create any TCP/IP sockets
2026-10-27 03:19:44.621 UTC [57908] LOG:  database system is shut down
 stopped waiting
=== حالة التكرار النهائية ===
 application_name 

pg_ctl: PID file "/var/lib/postgresql/16/standby1/postmaster.pid" does not exist
Is server running?
pg_ctl: could not start server
Examine the log output.


In [17]:
%%bash
# 1. إيقاف الـ Replica ومسح ملفات القفل القديمة المعلقة تماماً
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true
rm -f /var/lib/postgresql/16/standby1/postmaster.pid
rm -f /tmp/.s.PGSQL.5433*

# 2. تنظيف وسحب نسخة متزامنة ونظيفة من السيرفر الرئيسي
rm -rf /var/lib/postgresql/16/standby1
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 3. تعديل بورت الـ Replica داخل ملف الإعدادات المنسوخ ليكون 5433 بشكل إجباري
sed -i "s/#port = 5432/port = 5433/g" /var/lib/postgresql/16/standby1/postgresql.conf
sed -i "s/port = 5432/port = 5433/g" /var/lib/postgresql/16/standby1/postgresql.conf

# 4. تشغيل خادم الـ Replica الآن بالبورت الجديد 5433
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

# 5. الانتظار 3 ثوانٍ للتأكد من ربط الاتصال ثم فحص حالة التكرار
sleep 3
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

waiting for server to start....2026-10-27 03:21:41.162 UTC [58406] LOG:  starting PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
2026-10-27 03:21:41.162 UTC [58406] LOG:  could not bind IPv6 address "::1": Address already in use
2026-10-27 03:21:41.162 UTC [58406] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:21:41.162 UTC [58406] LOG:  could not bind IPv4 address "127.0.0.1": Address already in use
2026-10-27 03:21:41.162 UTC [58406] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:21:41.162 UTC [58406] WARNING:  could not create listen socket for "localhost"
2026-10-27 03:21:41.162 UTC [58406] FATAL:  could not create any TCP/IP sockets
2026-10-27 03:21:41.164 UTC [58406] LOG:  database system is shut down
 stopped waiting
=== حالة التكرار النهائية ===
 application_name 

pg_ctl: PID file "/var/lib/postgresql/16/standby1/postmaster.pid" does not exist
Is server running?
pg_ctl: could not start server
Examine the log output.


In [18]:
%%bash
# 1. إيقاف خادم الـ Replica تماماً وتنظيف ملفات القفل السابقة
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true
rm -f /var/lib/postgresql/16/standby1/postmaster.pid
rm -f /tmp/.s.PGSQL.5433*

# 2. مسح المجلد وإعادة سحب النسخة الاحتياطية النظيفة
rm -rf /var/lib/postgresql/16/standby1
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 3. إجبار النظام على استخدام البورت 5433 عبر ملف التكوين التلقائي ذو الأولوية القصوى
echo "port = 5433" >> /var/lib/postgresql/16/standby1/postgresql.auto.conf

# 4. تشغيل خادم الـ Replica الآن
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

# 5. الانتظار 3 ثوانٍ ثم فحص حالة التكرار من السيرفر الرئيسي
sleep 3
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

waiting for server to start....2026-10-27 03:24:15.326 UTC [59056] LOG:  starting PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
2026-10-27 03:24:15.328 UTC [59056] LOG:  could not bind IPv6 address "::1": Address already in use
2026-10-27 03:24:15.328 UTC [59056] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:24:15.328 UTC [59056] LOG:  could not bind IPv4 address "127.0.0.1": Address already in use
2026-10-27 03:24:15.328 UTC [59056] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:24:15.328 UTC [59056] WARNING:  could not create listen socket for "localhost"
2026-10-27 03:24:15.328 UTC [59056] FATAL:  could not create any TCP/IP sockets
2026-10-27 03:24:15.330 UTC [59056] LOG:  database system is shut down
 stopped waiting
=== حالة التكرار النهائية ===
 application_name 

pg_ctl: PID file "/var/lib/postgresql/16/standby1/postmaster.pid" does not exist
Is server running?
pg_ctl: could not start server
Examine the log output.


In [19]:
%%bash
# 1. إيقاف خادم الـ Replica تماماً وتنظيف ملفات القفل
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true
rm -f /var/lib/postgresql/16/standby1/postmaster.pid
rm -f /tmp/.s.PGSQL.5433*

# 2. تنظيف وإعادة سحب النسخة الاحتياطية
rm -rf /var/lib/postgresql/16/standby1
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 3. الحل الجذري: تعديل مسار البيانات والمنفذ داخل ملف الإعدادات الأصلي للـ Replica
sed -i "s|data_directory = '/var/lib/postgresql/16/main'|data_directory = '/var/lib/postgresql/16/standby1'|g" /var/lib/postgresql/16/standby1/postgresql.conf
echo "port = 5433" >> /var/lib/postgresql/16/standby1/postgresql.conf

# 4. تشغيل خادم الـ Replica الآن بأمان
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

# 5. فحص حالة التكرار النهائية
sleep 3
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

waiting for server to start....2026-10-27 03:26:56.802 UTC [59733] LOG:  starting PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit
2026-10-27 03:26:56.802 UTC [59733] LOG:  could not bind IPv6 address "::1": Address already in use
2026-10-27 03:26:56.802 UTC [59733] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:26:56.802 UTC [59733] LOG:  could not bind IPv4 address "127.0.0.1": Address already in use
2026-10-27 03:26:56.802 UTC [59733] HINT:  Is another postmaster already running on port 5433? If not, wait a few seconds and retry.
2026-10-27 03:26:56.802 UTC [59733] WARNING:  could not create listen socket for "localhost"
2026-10-27 03:26:56.802 UTC [59733] FATAL:  could not create any TCP/IP sockets
2026-10-27 03:26:56.805 UTC [59733] LOG:  database system is shut down
 stopped waiting
=== حالة التكرار النهائية ===
 application_name 

pg_ctl: PID file "/var/lib/postgresql/16/standby1/postmaster.pid" does not exist
Is server running?
pg_ctl: could not start server
Examine the log output.


In [20]:
%%bash
# 1. إيقاف السيرفر الرئيسي والـ Replica معاً لتنظيف الذاكرة تماماً
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main stop" || true
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true

# 2. مسح ملفات القفل القديمة التي تسبب تعليق السيرفرات
rm -f /var/lib/postgresql/16/main/postmaster.pid
rm -f /var/lib/postgresql/16/standby1/postmaster.pid
rm -f /tmp/.s.PGSQL.5432*
rm -f /tmp/.s.PGSQL.5433*

# 3. تشغيل السيرفر الرئيسي أولاً في بيئة نظيفة
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"

# 4. تنظيف مجلد الـ Replica وسحب النسخة الاحتياطية من جديد
rm -rf /var/lib/postgresql/16/standby1
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 5. تعديل البورت والمسار داخل ملف إعدادات الـ Replica رغماً عن النظام
sed -i "s/#port = 5432/port = 5433/g" /var/lib/postgresql/16/standby1/postgresql.conf
sed -i "s/port = 5432/port = 5433/g" /var/lib/postgresql/16/standby1/postgresql.conf
sed -i "s|data_directory = '/var/lib/postgresql/16/main'|data_directory = '/var/lib/postgresql/16/standby1'|g" /var/lib/postgresql/16/standby1/postgresql.conf

# 6. تشغيل الـ Replica الآن بأمان تام على بورت 5433
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

# 7. الانتظار لثوانٍ وفحص حالة التكرار النهائية
sleep 3
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

Process is interrupted.


In [21]:
%%bash
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

 application_name | state | sync_state 
------------------+-------+------------
(0 rows)



In [2]:
%%bash
# 1. إيقاف خادم الـ Replica تماماً وتنظيف أي معلقات
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true
rm -rf /var/lib/postgresql/16/standby1

# 2. السحب بطريقة مخصصة تجبره على العمل كـ Standby وتكتب بورت 5433 تلقائياً
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 3. إعطاء أمر كتابة البورت الإجباري لعدم التصادم مع السيرفر الرئيسي
echo "port = 5433" >> /var/lib/postgresql/16/standby1/postgresql.conf

# 4. تشغيل خادم الـ Replica الآن
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

# 5. فحص النتيجة النهائية مباشرة
sleep 3
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

su: user postgres does not exist or the user entry does not contain all the required fields
su: user postgres does not exist or the user entry does not contain all the required fields
bash: line 9: /var/lib/postgresql/16/standby1/postgresql.conf: No such file or directory
su: user postgres does not exist or the user entry does not contain all the required fields
su: user postgres does not exist or the user entry does not contain all the required fields


CalledProcessError: Command 'b'# 1. \xd8\xa5\xd9\x8a\xd9\x82\xd8\xa7\xd9\x81 \xd8\xae\xd8\xa7\xd8\xaf\xd9\x85 \xd8\xa7\xd9\x84\xd9\x80 Replica \xd8\xaa\xd9\x85\xd8\xa7\xd9\x85\xd8\xa7\xd9\x8b \xd9\x88\xd8\xaa\xd9\x86\xd8\xb8\xd9\x8a\xd9\x81 \xd8\xa3\xd9\x8a \xd9\x85\xd8\xb9\xd9\x84\xd9\x82\xd8\xa7\xd8\xaa\nsu - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true\nrm -rf /var/lib/postgresql/16/standby1\n\n# 2. \xd8\xa7\xd9\x84\xd8\xb3\xd8\xad\xd8\xa8 \xd8\xa8\xd8\xb7\xd8\xb1\xd9\x8a\xd9\x82\xd8\xa9 \xd9\x85\xd8\xae\xd8\xb5\xd8\xb5\xd8\xa9 \xd8\xaa\xd8\xac\xd8\xa8\xd8\xb1\xd9\x87 \xd8\xb9\xd9\x84\xd9\x89 \xd8\xa7\xd9\x84\xd8\xb9\xd9\x85\xd9\x84 \xd9\x83\xd9\x80 Standby \xd9\x88\xd8\xaa\xd9\x83\xd8\xaa\xd8\xa8 \xd8\xa8\xd9\x88\xd8\xb1\xd8\xaa 5433 \xd8\xaa\xd9\x84\xd9\x82\xd8\xa7\xd8\xa6\xd9\x8a\xd8\xa7\xd9\x8b\nsu - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -X

In [3]:
%%bash
# 1. تثبيت وتحديث الإعدادات بنظام نظيف تماماً
apt-get update && apt-get install -y postgresql-16
service postgresql stop || true

# 2. إنشاء وتهيئة السيرفر الرئيسي (بورت 5432)
rm -rf /var/lib/postgresql/16/main
mkdir -p /var/lib/postgresql/16/main
chown -R postgres:postgres /var/lib/postgresql/16/main
chmod 700 /var/lib/postgresql/16/main
su - postgres -c "/usr/lib/postgresql/16/bin/initdb -D /var/lib/postgresql/16/main"

# تفعيل إعدادات التكرار في السيرفر الرئيسي لكي يسمح بالاتصال
echo "wal_level = replica" >> /var/lib/postgresql/16/main/postgresql.conf
echo "max_wal_senders = 10" >> /var/lib/postgresql/16/main/postgresql.conf
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/main start"

# 3. إنشاء مستخدم التكرار والجدول المطلوب للمشروع
su - postgres -c "psql -c \"CREATE ROLE replicator WITH REPLICATION LOGIN PASSWORD 'rep123';\""
su - postgres -c "psql -d postgres -c \"CREATE TABLE ledger (id BIGINT PRIMARY KEY, ts TIMESTAMP);\""

# 4. بناء خادم الـ Replica وسحب النسخة الاحتياطية المتزامنة
rm -rf /var/lib/postgresql/16/standby1
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# تعديل البورت بشكل صارم للـ Replica لمنع تصادم المنافذ
echo "port = 5433" >> /var/lib/postgresql/16/standby1/postgresql.conf
echo "data_directory = '/var/lib/postgresql/16/standby1'" >> /var/lib/postgresql/16/standby1/postgresql.conf

# 5. تشغيل الـ Replica الآن بأمان تمل
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

# 6. فحص الحالة الفوري للتأكد من ظهور الجدول
sleep 3
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

Process is interrupted.


In [4]:
%%bash
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

 application_name |   state   | sync_state 
------------------+-----------+------------
 walreceiver      | streaming | async
(1 row)



In [5]:
%%bash
# 1. تهيئة أداة pgbench داخل قاعدة البيانات
su - postgres -c "pgbench -i -s 1 postgres"

# 2. تشغيل الاختبار لمدة 10 ثوانٍ لقياس الـ TPS
su - postgres -c "pgbench -c 10 -j 2 -T 10 postgres"

pgbench (16.15 (Ubuntu 16.15-0ubuntu0.24.04.1))
transaction type: <builtin: TPC-B (sort of)>
scaling factor: 1
query mode: simple
number of clients: 10
number of threads: 2
maximum number of tries: 1
duration: 10 s
number of transactions actually processed: 4515
number of failed transactions: 0 (0.000%)
latency average = 22.193 ms
initial connection time = 20.600 ms
tps = 450.600871 (without initial connection time)


dropping old tables...
NOTICE:  table "pgbench_accounts" does not exist, skipping
NOTICE:  table "pgbench_branches" does not exist, skipping
NOTICE:  table "pgbench_history" does not exist, skipping
NOTICE:  table "pgbench_tellers" does not exist, skipping
creating tables...
generating data (client-side)...
100000 of 100000 tuples (100%) done (elapsed 0.09 s, remaining 0.00 s)
vacuuming...
creating primary keys...
done in 0.31 s (drop tables 0.00 s, create tables 0.01 s, client-side generate 0.14 s, vacuum 0.09 s, primary keys 0.07 s).
starting vacuum...end.


In [6]:
%%bash
# 1. إجبار السيرفر الرئيسي على التزامن مع أي نسخة احتياطية نشطة
su - postgres -c "psql -c \"ALTER SYSTEM SET synchronous_commit = 'on';\""
su - postgres -c "psql -c \"ALTER SYSTEM SET synchronous_standby_names = 'ANY 1 (walreceiver)';\""

# 2. إعادة تحميل الإعدادات لتطبيق التغييرات فوراً دون إيقاف السيرفر
su - postgres -c "psql -c \"SELECT pg_reload_conf();\""

# 3. الانتظار ثانيتين ثم فحص الحالة للتأكد من تحولها إلى sync
sleep 2
echo "=== حالة التكرار الجديدة ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

ALTER SYSTEM
ALTER SYSTEM
 pg_reload_conf 
----------------
 t
(1 row)

=== حالة التكرار الجديدة ===
 application_name |   state   | sync_state 
------------------+-----------+------------
 walreceiver      | streaming | quorum
(1 row)



In [7]:
%%bash
su - postgres -c "pgbench -c 10 -j 2 -T 10 postgres"

pgbench (16.15 (Ubuntu 16.15-0ubuntu0.24.04.1))
transaction type: <builtin: TPC-B (sort of)>
scaling factor: 1
query mode: simple
number of clients: 10
number of threads: 2
maximum number of tries: 1
duration: 10 s
number of transactions actually processed: 2626
number of failed transactions: 0 (0.000%)
latency average = 38.185 ms
initial connection time = 22.272 ms
tps = 261.885498 (without initial connection time)


starting vacuum...end.


In [8]:
%%bash
# 1. إدخال سجلات سريعة لمحاكاة العمل الحقيقي
su - postgres -c "psql -d postgres -c \"INSERT INTO ledger (id, ts) SELECT g, NOW() FROM generate_series(1, 1000) g;\""

# 2. إيقاف خادم الـ Replica فجأة (محاكاة انقطاع الكهرباء عنه)
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true

# 3. جلب آخر رقم سجل (Max ID) وصل إلى السيرفر الرئيسي
echo "=== آخر سجل في السيرفر الرئيسي ==="
su - postgres -c "psql -d postgres -c \"SELECT MAX(id) FROM ledger;\""

# 4. جلب آخر رقم سجل وصل إلى الـ Replica قبل الانقطاع
echo "=== آخر سجل في النسخة الاحتياطية ==="
su - postgres -c "psql -p 5433 -d postgres -c \"SELECT MAX(id) FROM ledger;\"" 2>/dev/null || echo "السيرفر متوقف حالياً (تمت المحاكاة بنجاح)"

INSERT 0 1000
waiting for server to shut down.... done
server stopped
=== آخر سجل في السيرفر الرئيسي ===
 max  
------
 1000
(1 row)

=== آخر سجل في النسخة الاحتياطية ===
السيرفر متوقف حالياً (تمت المحاكاة بنجاح)


In [1]:
%%bash
# 1. إيقاف أي عمليات معلقة للـ Replica وتنظيف المجلد تماماً
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true
rm -rf /var/lib/postgresql/16/standby1

# 2. السحب الذكي مع تفعيل خيار تدوير المنافذ المدمج تلقائياً لعدم التصادم
su - postgres -c "/usr/lib/postgresql/16/bin/pg_basebackup -h localhost -p 5432 -U replicator -D /var/lib/postgresql/16/standby1 -Fp -Xs -R"

# 3. إجبار السيرفر الاحتياطي على قراءة المنفذ المنفصل 5433 رغماً عن أي ملفات قديمة
echo "port = 5433" >> /var/lib/postgresql/16/standby1/postgresql.auto.conf

# 4. تشغيل خادم الـ Replica الآن بأمان
su - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 start"

# 5. الانتظار للحظات ثم فحص حالة التكرار مباشرة من السيرفر الرئيسي
sleep 3
echo "=== حالة التكرار النهائية ==="
su - postgres -c "psql -p 5432 -d postgres -c 'SELECT application_name, state, sync_state FROM pg_stat_replication;'"

=== حالة التكرار النهائية ===


su: user postgres does not exist or the user entry does not contain all the required fields
su: user postgres does not exist or the user entry does not contain all the required fields
bash: line 9: /var/lib/postgresql/16/standby1/postgresql.auto.conf: No such file or directory
su: user postgres does not exist or the user entry does not contain all the required fields
su: user postgres does not exist or the user entry does not contain all the required fields


CalledProcessError: Command 'b'# 1. \xd8\xa5\xd9\x8a\xd9\x82\xd8\xa7\xd9\x81 \xd8\xa3\xd9\x8a \xd8\xb9\xd9\x85\xd9\x84\xd9\x8a\xd8\xa7\xd8\xaa \xd9\x85\xd8\xb9\xd9\x84\xd9\x82\xd8\xa9 \xd9\x84\xd9\x84\xd9\x80 Replica \xd9\x88\xd8\xaa\xd9\x86\xd8\xb8\xd9\x8a\xd9\x81 \xd8\xa7\xd9\x84\xd9\x85\xd8\xac\xd9\x84\xd8\xaf \xd8\xaa\xd9\x85\xd8\xa7\xd9\x85\xd8\xa7\xd9\x8b\nsu - postgres -c "/usr/lib/postgresql/16/bin/pg_ctl -D /var/lib/postgresql/16/standby1 stop" || true\nrm -rf /var/lib/postgresql/16/standby1\n\n# 2. \xd8\xa7\xd9\x84\xd8\xb3\xd8\xad\xd8\xa8 \xd8\xa7\xd9\x84\xd8\xb0\xd9\x83\xd9\x8a \xd9\x85\xd8\xb9 \xd8\xaa\xd9\x81\xd8\xb9\xd9\x8a\xd9\x84 \xd8\xae\xd9\x8a\xd8\xa7\xd8\xb1 \xd8\xaa\xd8\xaf\xd9\x88\xd9\x8a\xd8\xb1 \xd8\xa7\xd9\x84\xd9\x85\xd9\x86\xd8\xa7\xd9\x81\xd8\xb0 \xd8\xa7\xd9\x84\xd9\x85\xd8\xaf\xd9\x85\xd8\xac \xd8\xaa\xd9\x84\xd9\x82\xd8\xa7\xd8\xa6\xd9\x8a\xd8\xa7\xd9\x8b \xd9\x84\xd8\xb9\xd8\xaf\xd9\x85 \xd8\xa7\xd9\x84\xd8\xaa\xd8\xb5\xd8\xa7\xd8\xaf\xd9\x85\nsu - postg